## RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

### Objective
Compare dense baseline retrieval with cross-encoder re-ranking
using the same PDF corpus, embedding model, and answer-generation prompt.

### Experimental Setup
- Document: source.pdf
- Dense retriever: sentence-transformers/all-MiniLM-L6-v2
- Cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
- Initial retrieval: Top 10 chunks
- Baseline context: Top 3 chunks
- Re-ranked context: Top 3 chunks after re-ranking

### Experiments
1. PDF extraction and chunking
2. Dense embedding and vector indexing
3. Baseline retrieval
4. Cross-encoder re-ranking
5. Failure-mode analysis
6. Latency profiling
7. Results and discussion

In [8]:

import os
import json
import time
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

from dotenv import load_dotenv
from google import genai


from IPython.display import display, Markdown
load_dotenv()

/home/arceus51/miniforge3/envs/experiments/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [9]:
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("../data/source.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found at {PDF_PATH.resolve()}. "
        "Place your document at data/source.pdf."
    )

reader = PdfReader(str(PDF_PATH))

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        pages.append({
            "page": page_number,
            "text": text,
        })

print(f"Total PDF pages: {len(reader.pages)}")
print(f"Pages with extractable text: {len(pages)}")

if pages:
    print("\nFirst page with extractable text:")
    print(f"Page number: {pages[0]['page']}")
    print(pages[0]["text"][:1500])
else:
    print("No extractable text found. Check whether the PDF is scanned.")

Total PDF pages: 187
Pages with extractable text: 187

First page with extractable text:
Page number: 1
 
 
 
 
 
 
 
 
 
B.Tech II Year IV sem 
Basic of Python 
Programming 
Prepared by S.VENKATA LAKSHMI .,M.TECH 
Course Code:20AES0509 
 


In [10]:
# Support execution from either the project root or notebooks/
candidates = [Path("data/source.pdf"),Path("../data/source.pdf"),]

PDF_PATH = next((p for p in candidates if p.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        "Could not find data/source.pdf. Check your project directory."
    )

reader = PdfReader(str(PDF_PATH))
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text,
        })

print(f"PDF: {PDF_PATH.resolve()}")
print(f"Total pages: {len(reader.pages)}")
print(f"Pages with text: {len(pages)}")
print(
    "Total extracted characters:",
    sum(len(p["text"]) for p in pages),
)

PDF: /home/arceus51/Projects/rag-baseline-reranking/data/source.pdf
Total pages: 187
Pages with text: 187
Total extracted characters: 340821


In [11]:
# splitting the documents into chunks for embedding and retrieval
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    add_start_index=True,
    separators=["\n\n", "\n", " ", ""],
)

documents = []

for page in pages:
    page_documents = splitter.create_documents(
        texts=[page["text"]],
        metadatas=[
            {
                "page": page["page"],
                "source": PDF_PATH.name,
            }
        ],
    )

    for document in page_documents:
        document.metadata["chunk_id"] = (
            f"chunk_{len(documents):05d}"
        )
        documents.append(document)

print("Total chunks:", len(documents))

print("\nFirst chunk:")
print("ID:", documents[0].metadata["chunk_id"])
print("Page:", documents[0].metadata["page"])
print("Characters:", len(documents[0].page_content))
print(documents[0].page_content[:500])

Total chunks: 515

First chunk:
ID: chunk_00000
Page: 1
Characters: 114
B.Tech II Year IV sem 
Basic of Python 
Programming 
Prepared by S.VENKATA LAKSHMI .,M.TECH 
Course Code:20AES0509


In [12]:
# generate embeddings and building the FAISS index 

import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

# Dense bi-encoder: embeds passages and queries independently
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

texts = [document.page_content for document in documents]

embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

# Normalized vectors + inner product = cosine similarity
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embedding shape:", embeddings.shape)
print("Vectors indexed:", index.ntotal)
print("Vector dimensions:", embeddings.shape[1])

Batches: 100%|██████████| 17/17 [00:07<00:00,  2.41it/s]

Embedding shape: (515, 384)
Vectors indexed: 515
Vector dimensions: 384


In [13]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

faiss.write_index(
    index,
    str(OUTPUT_DIR / "faiss.index"),
)

with open(OUTPUT_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for document in documents:
        record = {
            "chunk_id": document.metadata["chunk_id"],
            "page": document.metadata["page"],
            "source": document.metadata["source"],
            "start_index": document.metadata.get("start_index"),
            "text": document.page_content,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved FAISS index and chunk metadata.")

Saved FAISS index and chunk metadata.


In [16]:
def dense_retrieve(query: str, k: int = 10) -> list[dict]:
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    k = min(k, index.ntotal)
    scores, indices = index.search(query_embedding, k)

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        document = documents[int(idx)]

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_id": document.metadata["chunk_id"],
            "page": document.metadata["page"],
            "text": document.page_content,
        })

    return results


query = "How does a Python function return a value to the caller?"

results = dense_retrieve(query, k=10)

for result in results:
    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']:.4f} | "
        f"Page: {result['page']} | "
        f"ID: {result['chunk_id']}"
    )
    print(result["text"][:350])


Rank 1 | Score: 0.6912 | Page: 16 | ID: chunk_00030
(expression) exits a function, optionally passing back a value to the caller. A return statement 
with no args is the same as return None. 
Significance of Indentation (Space) in Python: 
 
Before we get familiarize with Python functions, it is important that we understand the 
indentation rule to declare Python functions and these rules are applic

Rank 2 | Score: 0.6594 | Page: 45 | ID: chunk_00087
Your 
 
o/p: 
enter value: 
42 
The value of c =52 
 
 
 
 
 
 
 
 
Chapter-3 
Fruitful Functions 
 
1. Return values: 
The return keyword is to exit a function and return a value. 
Syntax: 
Return or return value: 
For ex : 
def myfunction(): 
return 3+3 
print("Hello, World!")

Rank 3 | Score: 0.6574 | Page: 22 | ID: chunk_00038
destroyed once we return from the function. Hence, a function does not remember the value 
of a variable from its previous calls.Here is an example to illustrate the scope of a variable 
inside a function. 
def

### Pipeline 2 - Cross-encoder re-ranking 

In [17]:
# ============================================================
# Cross-Encoder Re-Ranker
# ============================================================

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(
    RERANKER_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print("Cross-encoder loaded successfully!")
print(f"Model: {RERANKER_MODEL}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15904.15it/s]


Cross-encoder loaded successfully!
Model: cross-encoder/ms-marco-MiniLM-L-6-v2


In [18]:
# ============================================================
# Re-rank the Top 10 Candidates
# ============================================================

def rerank_results(
    query: str,
    candidates: list[dict],
) -> list[dict]:
    """
    Re-rank dense-retrieval candidates using a cross-encoder.

    Each candidate is scored jointly with the query.
    Candidates are then sorted by cross-encoder score.
    """

    if not candidates:
        return []

    query_chunk_pairs = [
        (query, candidate["text"])
        for candidate in candidates
    ]

    scores = reranker.predict(
        query_chunk_pairs,
        batch_size=16,
        show_progress_bar=False,
    )

    reranked = []

    for candidate, score in zip(candidates, scores):
        result = candidate.copy()

        # Preserve the dense retrieval rank for comparison.
        result["original_rank"] = candidate["rank"]

        # Store the cross-encoder score.
        result["cross_encoder_score"] = float(score)

        reranked.append(result)

    # Highest cross-encoder score first.
    reranked.sort(
        key=lambda item: item["cross_encoder_score"],
        reverse=True,
    )

    # Assign new ranks after re-ranking.
    for new_rank, result in enumerate(reranked, start=1):
        result["rerank_rank"] = new_rank

    return reranked

In [19]:
query = "How does a Python function return a value to the caller?"

# Pipeline 1: Dense retrieval
baseline_results = dense_retrieve(query, k=10)

# Pipeline 2: Re-rank those exact same 10 candidates
reranked_results = rerank_results(
    query=query,
    candidates=baseline_results,
)

# Verify that both pipelines used the same candidates.
assert {
    item["chunk_id"] for item in baseline_results
} == {
    item["chunk_id"] for item in reranked_results
}

# Compare rank changes.
comparison = pd.DataFrame([
    {
        "Chunk ID": item["chunk_id"],
        "Page": item["page"],
        "Baseline Rank": item["original_rank"],
        "Re-ranked Rank": item["rerank_rank"],
        "Rank Change": item["original_rank"] - item["rerank_rank"],
        "Cross-Encoder Score": round(
            item["cross_encoder_score"], 4
        ),
    }
    for item in reranked_results
])

display(
    comparison.sort_values("Re-ranked Rank").reset_index(drop=True)
)

,Chunk ID,Page,Baseline Rank,Re-ranked Rank,Rank Change,Cross-Encoder Score
0,chunk_00031,17,5,1,4,8.9074
1,chunk_00030,16,1,2,-1,7.1708
2,chunk_00029,16,4,3,1,6.8140
3,chunk_00033,19,8,4,4,6.0342
4,chunk_00037,22,9,5,4,3.4022
5,chunk_00038,22,3,6,-3,2.3251
6,chunk_00425,157,6,7,-1,1.4958
7,chunk_00087,45,2,8,-6,0.6507
8,chunk_00059,31,10,9,1,0.4177
9,chunk_00345,133,7,10,-3,-1.1963


In [20]:
print("=" * 80)
print("BASELINE RETRIEVAL — TOP 3")
print("=" * 80)

for item in baseline_results[:3]:
    print(
        f"\nRank {item['rank']} | "
        f"{item['chunk_id']} | Page {item['page']}"
    )
    print(item["text"][:500])


print("\n" + "=" * 80)
print("CROSS-ENCODER RE-RANKING — TOP 3")
print("=" * 80)

for item in reranked_results[:3]:
    print(
        f"\nRank {item['rerank_rank']} | "
        f"{item['chunk_id']} | "
        f"Original Rank {item['original_rank']} | "
        f"Page {item['page']}"
    )
    print(item["text"][:500])

BASELINE RETRIEVAL — TOP 3

Rank 1 | chunk_00030 | Page 16
(expression) exits a function, optionally passing back a value to the caller. A return statement 
with no args is the same as return None. 
Significance of Indentation (Space) in Python: 
 
Before we get familiarize with Python functions, it is important that we understand the 
indentation rule to declare Python functions and these rules are applicable to other elements 
of Python as well like declaring conditions, loops or variable. 
Python follows a particu lar style of indentation to define t

Rank 2 | chunk_00087 | Page 45
Your 
 
o/p: 
enter value: 
42 
The value of c =52 
 
 
 
 
 
 
 
 
Chapter-3 
Fruitful Functions 
 
1. Return values: 
The return keyword is to exit a function and return a value. 
Syntax: 
Return or return value: 
For ex : 
def myfunction(): 
return 3+3 
print("Hello, World!")

Rank 3 | chunk_00038 | Page 22
destroyed once we return from the function. Hence, a function does not remember the value 
of a 